# Thêm Thư Viện

In [1]:
import pyodbc
import pandas as pd
import numpy as np

# Tạo kết nối

In [2]:
conn_dwh_library = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=192.168.150.6;' # Địa chỉ IP của SQL Server
    'DATABASE=DWH_Lib;' # Tên cơ sở dữ liệu
    'UID=itc;'              # Tên đăng nhập
    'PWD=spkt@2025;'
)

## Đọc data từ CSV

In [3]:
df_data_khoa = pd.read_csv("./data_khoa.csv")
print(df_data_khoa)

    ID_Khoa                              TenKhoa
0         1              Khoa Cơ khí Chế tạo máy
1         2                 Khoa Cơ khí Động lực
2         3  Khoa Công nghệ Hóa học và Thực phẩm
3         4           Khoa Thời trang và Du lịch
4         5             Khoa Công nghệ Thông tin
5         6          Khoa Đào tạo chất lượng cao
6         7                  Khoa Điện - Điện tử
7         8               Khoa In - Truyền thông
8         9               Khoa Khoa học Ứng dụng
9        10                         Khoa Kinh tế
10       11               Khoa Chính trị và Luật
11       12                       Khoa Ngoại ngữ
12       13                        Khoa Xây dựng
13       14                Viện Sư phạm Kỹ thuật
14       15                 Khoa Đào tạo Quốc tế


## Xử lý data

In [4]:
# Tạo hàng dữ liệu giả lập cho khoa không xác định
new_row = pd.DataFrame({'ID_Khoa': [0], 'TenKhoa': ['(Không xác định)']})
df_data_khoa = pd.concat([df_data_khoa, new_row], ignore_index=True) # Thêm vào dataset
df_data_khoa = df_data_khoa.sort_values(by='ID_Khoa', ascending=True).reset_index(drop=True) # sắp xếp lại cho dễ nhìn
print(df_data_khoa)

    ID_Khoa                              TenKhoa
0         0                     (Không xác định)
1         1              Khoa Cơ khí Chế tạo máy
2         2                 Khoa Cơ khí Động lực
3         3  Khoa Công nghệ Hóa học và Thực phẩm
4         4           Khoa Thời trang và Du lịch
5         5             Khoa Công nghệ Thông tin
6         6          Khoa Đào tạo chất lượng cao
7         7                  Khoa Điện - Điện tử
8         8               Khoa In - Truyền thông
9         9               Khoa Khoa học Ứng dụng
10       10                         Khoa Kinh tế
11       11               Khoa Chính trị và Luật
12       12                       Khoa Ngoại ngữ
13       13                        Khoa Xây dựng
14       14                Viện Sư phạm Kỹ thuật
15       15                 Khoa Đào tạo Quốc tế


## Load data

### [Nếu cần] Clear bảng

In [5]:
cursor = conn_dwh_library.cursor()
truncate_query = "DELETE FROM olap.DIM_Khoa"
cursor.execute(truncate_query)
conn_dwh_library.commit()
cursor.close()

### Load data vào bảng Dim_Khoa

In [6]:
cursor_dwh = conn_dwh_library.cursor()
insert_query = """
                INSERT INTO olap.DIM_Khoa (ID_khoa, Ten_khoa) 
                VALUES (?, ?)
                """
for index, row in df_data_khoa.iterrows():
    values = (row['ID_Khoa'], 
              row['TenKhoa'])
    cursor_dwh.execute(insert_query, values)
conn_dwh_library.commit()